In [ ]:
!pip install arxiv transformers

#Rawdata to Local Test

In [ ]:
import arxiv
import os
from pathlib import Path

def download_qfin_papers(category="q-fin", max_papers=100, output_dir="./papers"):
    Path(output_dir).mkdir(exist_ok=True)

    client = arxiv.Client(page_size=100, delay_seconds=3)
    search = arxiv.Search(
        query=f"cat:{category}",
        max_results=max_papers,
        sort_by=arxiv.SortCriterion.SubmittedDate
    )

    for result in client.results(search):
        try:
            filename = f"{result.entry_id.split('/abs/')[-1].replace('/', '_')}.pdf"
            result.download_pdf(dirpath=output_dir, filename=filename)
            print(f"✓ {result.title[:60]}...")
        except Exception as e:
            print(f"✗ Failed: {e}")

# Download papers
download_qfin_papers(category="q-fin.MF", max_papers=50)

# Download rawdata to GCS Storage bucket

In [2]:
import arxiv
import os
from pathlib import Path

def download_qfin_papers(category="q-fin", max_papers=100, output_dir="./papers"):
    Path(output_dir).mkdir(exist_ok=True)

    client = arxiv.Client(page_size=100, delay_seconds=3)
    search = arxiv.Search(
        query=f"cat:{category}",
        max_results=max_papers,
        sort_by=arxiv.SortCriterion.SubmittedDate
    )

    for result in client.results(search):
        try:
            filename = f"{result.entry_id.split('/abs/')[-1].replace('/', '_')}.pdf"
            result.download_pdf(dirpath=output_dir, filename=filename)
            print(f"✓ {result.title[:60]}...")
        except Exception as e:
            print(f"✗ Failed: {e}")

# Download papers
download_qfin_papers(category="q-fin.MF", max_papers=50)

✓ The Omniscient, yet Lazy, Investor...
✓ Extended HJB Equation for Mean-Variance Stopping Problem: Va...
✓ Inverse Behavioral Optimization of QALY-Based Incentive Syst...
✓ Goal-based portfolio selection with fixed transaction costs...
✓ Jump risk premia in the presence of clustered jumps...
✓ Portfolio selection with exogenous and endogenous transactio...
✓ Consumption-Investment Problem in Rank-Based Models...
✓ Multivariate Variance Swap Using Generalized Variance Method...
✓ Simultaneously Solving Infinitely Many LQ Mean Field Games I...
✓ Semi-analytical pricing of American options with hybrid divi...
✓ A Mixed-Form PINNS (MF-PINNS) For Solving The Coupled Stokes...
✓ A Topological Approach to Parameterizing Deep Hedging Networ...
✓ Martingale theory for Dynkin games with asymmetric informati...
✓ On the short-time behaviour of up-and-in barrier options usi...
✓ On Time-subordinated Brownian Motion Processes for Financial...
✓ The Variance-Gamma Process for Option Pricing...
✓ On

In [5]:
!cwd

'cwd' is not recognized as an internal or external command,
operable program or batch file.


In [9]:
import random

# List all downloaded PDF files
paper_files = [f for f in os.listdir("./papers") if f.endswith(".pdf")]

# Select a random paper
if paper_files:
    random_paper_path = os.path.join("./papers", random.choice(paper_files))
    print(f"Loading random paper: {random_paper_path}")

    # need a PDF reader librar to extract text content from the PDF file.
else:
    print("No papers found in the ./papers directory.")

Loading random paper: ./papers\2510.11829v2.pdf


import arxiv
import os
from pathlib import Path
from google.cloud import storage
from dotenv import load_dotenv
import tempfile
from typing import Optional

# Load environment variables from .env file
load_dotenv()

In [ ]:

class ArxivGCSDownloader:
    """Download arXiv papers and upload them to Google Cloud Storage"""
    
    def __init__(self, bucket_name: Optional[str] = None, gcs_folder: str = "papers"):
        """
        Initialize the downloader with GCS credentials
        
        Args:
            bucket_name: GCS bucket name (defaults to env variable GCS_BUCKET_NAME)
            gcs_folder: Folder path within the bucket (default: "papers")
        """
        # Load GCS configuration from environment
        self.bucket_name = bucket_name or os.getenv("GCS_BUCKET_NAME")
        self.gcs_folder = gcs_folder
        
        # Initialize GCS client
        # GOOGLE_APPLICATION_CREDENTIALS env var should point to service account JSON
        credentials_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
        if credentials_path and not os.path.exists(credentials_path):
            raise FileNotFoundError(f"Credentials file not found: {credentials_path}")
        
        self.storage_client = storage.Client()
        self.bucket = self.storage_client.bucket(self.bucket_name)
        
        print(f"Initialized GCS client for bucket: {self.bucket_name}")
        print(f"Papers will be uploaded to: gs://{self.bucket_name}/{self.gcs_folder}/")
    
    def download_and_upload_papers(
        self, 
        category: str = "q-fin", 
        max_papers: int = 100,
        local_temp_storage: bool = False,
        temp_dir: str = "./papers_temp"
    ):
        """
        Download papers from arXiv and upload to GCS
        
        Args:
            category: arXiv category (e.g., "q-fin.MF")
            max_papers: Maximum number of papers to download
            local_temp_storage: If True, keep local copies in temp_dir
            temp_dir: Directory for temporary local storage
        """
        # Create temp directory if keeping local copies
        if local_temp_storage:
            Path(temp_dir).mkdir(exist_ok=True)
        
        # Initialize arXiv client
        client = arxiv.Client(page_size=100, delay_seconds=3)
        search = arxiv.Search(
            query=f"cat:{category}",
            max_results=max_papers,
            sort_by=arxiv.SortCriterion.SubmittedDate
        )
        
        download_count = 0
        upload_count = 0
        error_count = 0
        
        print(f"\nStarting download for category: {category}")
        print(f"Maximum papers: {max_papers}\n")
        
        for result in client.results(search):
            try:
                # Generate filename from paper ID
                filename = f"{result.entry_id.split('/abs/')[-1].replace('/', '_')}.pdf"
                
                # Use temporary directory for download
                with tempfile.TemporaryDirectory() as tmpdirname:
                    # Download paper to temp location
                    temp_path = os.path.join(tmpdirname, filename)
                    result.download_pdf(dirpath=tmpdirname, filename=filename)
                    download_count += 1
                    
                    # Upload to GCS
                    gcs_path = f"{self.gcs_folder}/{filename}"
                    blob = self.bucket.blob(gcs_path)
                    
                    # Check if file already exists
                    if blob.exists():
                        print(f"⊙ Already exists: {result.title[:50]}... (skipping)")
                        continue
                    
                    # Upload with metadata
                    blob.metadata = {
                        'title': result.title,
                        'authors': ', '.join([author.name for author in result.authors]),
                        'published': str(result.published),
                        'arxiv_id': result.entry_id,
                        'categories': ', '.join(result.categories)
                    }
                    
                    blob.upload_from_filename(temp_path)
                    upload_count += 1
                    
                    print(f"✓ Uploaded: {result.title[:50]}...")
                    print(f"  → gs://{self.bucket_name}/{gcs_path}")
                    
                    # Optionally keep local copy
                    if local_temp_storage:
                        local_path = os.path.join(temp_dir, filename)
                        with open(temp_path, 'rb') as src, open(local_path, 'wb') as dst:
                            dst.write(src.read())
                
            except Exception as e:
                error_count += 1
                print(f"✗ Failed: {result.title[:50] if hasattr(result, 'title') else 'Unknown'}")
                print(f"  Error: {e}")
        
        # Print summary
        print(f"\n{'='*60}")
        print(f"Download Summary:")
        print(f"  Downloaded: {download_count}")
        print(f"  Uploaded to GCS: {upload_count}")
        print(f"  Errors: {error_count}")
        print(f"  GCS Location: gs://{self.bucket_name}/{self.gcs_folder}/")
        print(f"{'='*60}")
        
        return {
            'downloaded': download_count,
            'uploaded': upload_count,
            'errors': error_count
        }
    
    def list_uploaded_papers(self, max_results: int = 10):
        """List papers in the GCS bucket"""
        print(f"\nListing papers in gs://{self.bucket_name}/{self.gcs_folder}/")
        blobs = self.storage_client.list_blobs(
            self.bucket_name, 
            prefix=self.gcs_folder,
            max_results=max_results
        )
        
        for blob in blobs:
            if blob.name.endswith('.pdf'):
                print(f"  • {blob.name}")
                if blob.metadata:
                    print(f"    Title: {blob.metadata.get('title', 'N/A')[:60]}")
        
# Example usage
if __name__ == "__main__":
    # Initialize downloader (uses environment variables)
    downloader = ArxivGCSDownloader()
    
    # Download and upload papers
    stats = downloader.download_and_upload_papers(
        category="q-fin.MF",
        max_papers=50,
        local_temp_storage=False  # Set to True to keep local copies
    )
    
    # List some uploaded papers
    print("\n")
    downloader.list_uploaded_papers(max_results=10)

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer

# Define the model name you want to fine-tune
model_name = "meta-llama/Llama-2-7b-hf"

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


# Set up a dummy dataset for demonstration purposes
# In a real scenario,  load and preprocess your research paper dataset here
# Make sure dataset is in a format compatible with the Hugging Face Traine

In [ ]:

dummy_dataset = ["This is a research paper about machine learning.", "Another paper discussing natural language processing."]

# Tokenize the dataset (a more robust tokenization would be needed for a real dataset)
tokenized_dataset = tokenizer(dummy_dataset, padding=True, truncation=True, return_tensors="pt")

# Convert to a format suitable for the Trainer
class DummyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings.input_ids)



train

In [ ]:
train_dataset = DummyDataset(tokenized_dataset)

# Define training arguments (you would customize these)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=10_000,
    save_total_limit=2,
)

# Initialize the Trainer (you would add your dataset and other parameters here)
# The trainer requires a dataset, which we've created a dummy for.
# In a real case, you would pass your preprocessed research paper dataset.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

# The fine-tuning process would start with trainer.train()
# For this basic setup, we are just showing the initialization.
# trainer.train()